# 🐍 AI Travel Agent with Error Handling - Microsoft Agent Framework (Python)

## 📋 Scenario Overview

This notebook demonstrates **production-ready error handling** in AI agents using the Microsoft Agent Framework. We'll build a flight booking agent that gracefully handles service failures by implementing backup systems and retry logic.

**Key Concepts:**
- 🛡️ **Error Handling**: Gracefully handle service failures with fallback mechanisms
- 🔄 **Retry Logic**: Automatically retry failed operations with backup services
- 📊 **Service Monitoring**: Simulate real-world API failures and recovery
- 🎯 **Resilience**: Build agents that continue functioning despite component failures

## 🏗️ Technical Implementation

### Core Components
- **Agent Framework**: Python implementation of Microsoft's agent orchestration
- **Azure OpenAI**: GPT-4o-mini for natural language understanding
- **Microsoft Entra ID**: Secure keyless authentication
- **Tool Functions**: Primary and backup flight services

### Error Handling Flow
```
User Request → get_flight_times() → HTTP 404 Error
                     ↓
           Agent detects error
                     ↓
      get_flight_times_backup() → Success!
```

## ⚙️ Prerequisites

**Required:**
```bash
pip install agent-framework-core azure-identity python-dotenv
```

**Environment Variables (.env):**
```env
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=gpt-4o-mini
AZURE_OPENAI_API_VERSION=2024-10-21
```

**Azure RBAC:** "Cognitive Services OpenAI User" role required

Let's build a resilient agent! 🌟

In [ ]:
# Package check - Install manually if needed: pip install agent-framework-core -U
try:
    import agent_framework
    print("✓ agent-framework-core is installed")
except ImportError:
    print("❌ Please install: pip install agent-framework-core -U")
    raise

In [ ]:
# 📦 Import Required Libraries
import os
from dotenv import load_dotenv
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import InteractiveBrowserCredential
from typing import Annotated
from IPython.display import display, HTML
import json

In [ ]:
# 🔧 Load Environment Variables
load_dotenv(override=True)

# Verify configuration
print(f"✓ AZURE_OPENAI_ENDPOINT: {os.environ.get('AZURE_OPENAI_ENDPOINT')}")
print(f"✓ AZURE_OPENAI_CHAT_DEPLOYMENT_NAME: {os.environ.get('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")
print(f"✓ AZURE_OPENAI_API_VERSION: {os.environ.get('AZURE_OPENAI_API_VERSION')}")
print(f"AZURE_OPENAI_API_KEY: {'⚠️  SET (removing for Entra ID auth)' if os.environ.get('AZURE_OPENAI_API_KEY') else '✓ Not set (using Entra ID)'}")

# Critical: Remove API key if present to force Entra ID authentication
if os.environ.get('AZURE_OPENAI_API_KEY'):
    del os.environ['AZURE_OPENAI_API_KEY']
    print("✓ API key removed - using Entra ID authentication")

In [ ]:
# 🎲 Tool Functions: Destination and Flight Information
# These functions simulate a primary service that fails and a backup service

def get_destinations() -> str:
    """Provides a list of vacation destinations.
    
    Returns:
        str: List of available vacation destinations
    """
    return """
    Barcelona, Spain
    Paris, France
    Berlin, Germany
    Tokyo, Japan
    New York, USA
    """

def get_flight_times(destination: str) -> str:
    """Provides available flight times for a destination.
    
    Args:
        destination: The destination to check flight times for
    
    Returns:
        str: Flight times or error message if service is down
    """
    # Simulate service failure
    return "HTTP ERROR 404: Flight times service is currently unavailable."

def get_flight_times_backup(destination: str) -> str:
    """Backup function that provides available flight times for a destination.
    
    Args:
        destination: The destination to check flight times for
    
    Returns:
        str: Flight times for the specified destination
    """
    flight_times = {
        "Barcelona": ["08:30 AM", "02:15 PM", "10:45 PM"],
        "Paris": ["06:45 AM", "12:30 PM", "07:15 PM"],
        "Berlin": ["07:20 AM", "01:45 PM", "09:30 PM"],
        "Tokyo": ["11:00 AM", "05:30 PM", "11:55 PM"],
        "New York": ["05:15 AM", "03:00 PM", "08:45 PM"]
    }
    
    # Extract just the city name from input that might contain country
    city = destination.split(',')[0].strip()
    
    if city in flight_times:
        times = ", ".join(flight_times[city])
        return f"Flight times for {city}: {times}"
    else:
        return f"No flight information available for {city}."

print("✓ Tool functions defined: get_destinations(), get_flight_times(), get_flight_times_backup()")

In [ ]:
# 🔗 Create Azure OpenAI Chat Client with Browser Authentication
# Uses Microsoft Entra ID for secure, keyless authentication

print("🔐 Authenticating with Azure...")
print("   A browser window will open for you to sign in")

# Create credential that will open browser for authentication
credential = InteractiveBrowserCredential()

# Create Azure OpenAI client
openai_chat_client = AzureOpenAIChatClient(
    credential=credential,
    endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION")
)

print(f"✓ Connected to Azure OpenAI")
print(f"  Endpoint: {os.environ.get('AZURE_OPENAI_ENDPOINT')}")
print(f"  Deployment: {os.environ.get('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")

In [ ]:
# 🤖 Create the Travel Agent with Error Handling Instructions
# The agent instructions explicitly tell it to handle errors and use backup services

AGENT_INSTRUCTIONS = """
You are Flight Booking Agent that provides information about available flights and gives travel activity suggestions when asked.
Travel activity suggestions should be specific to customer, location and amount of time at location.

You have access to the following tools to help users plan their trips:
1. get_destinations: Returns a list of available vacation destinations that users can choose from.
2. get_flight_times: Provides available flight times for specific destinations.
3. get_flight_times_backup: Backup function that provides available flight times when the primary service is down.

Your process for assisting users:
- When users inquire about flight booking, book the earliest flight available for the destination they choose using get_flight_times.
- If get_flight_times returns an error message, immediately use get_flight_times_backup with the same destination parameter to retrieve flight information.
- Since you do not have access to a booking system, DO NOT ask to proceed with booking, just assume you have booked the flight.
- Use any past conversation history to understand user preferences and consider them when making suggestions on flights and activities. When making a suggestion, be very clear on why you are making this suggestion if based on a user preference.

Guidelines:
- Use the exact destination names when using tools (Barcelona, Paris, Berlin, Tokyo, New York)
- Respond in a helpful and enthusiastic manner about travel possibilities
- Always seek feedback to ensure your suggestions meet the user's expectations
- Acknowledge when a request falls outside your capabilities
- For better formatting, always display flight times in a list format
- When giving any timed suggestions, reflect if the time frames are reasonable. Respond again if not.
- If the flight times service is down, inform the user that you're using backup flight data while maintaining a positive tone.

Your goal is to help users explore vacation options efficiently and make informed travel decisions by understanding their preferences and providing tailored recommendations.
"""

agent = ChatAgent(
    chat_client=openai_chat_client,
    instructions=AGENT_INSTRUCTIONS,
    tools=[get_destinations, get_flight_times, get_flight_times_backup]
)

print("✓ Travel agent created with error handling and backup service capabilities")

In [ ]:
# 💬 Run the Agent: Demonstrate Error Handling
# Watch how the agent detects the error from get_flight_times() and automatically uses get_flight_times_backup()

user_inputs = [
    "Book me a flight to Barcelona",
]

print("🎯 Testing error handling with service failure...\n")

async def run_conversation():
    for user_input in user_inputs:
        # Display user message
        html_output = (
            f"<div style='margin-bottom:10px'>" 
            f"<div style='font-weight:bold; color:#0066cc;'>👤 User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )
        
        # Get agent response (await the async call)
        response = await agent.run(user_input)
        
        # Display agent response
        html_output += (
            f"<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold; color:#009900;'>🤖 TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{response}</div></div><hr>"
        )
        
        display(HTML(html_output))

# Run the async conversation
await run_conversation()

print("\n✅ Error handling demonstrated! The agent successfully used the backup service.")

## 🎓 Key Takeaways: Production-Ready Error Handling

**What We Demonstrated:**
1. ✅ **Service Failure Detection**: Agent recognized HTTP 404 error from primary service
2. ✅ **Automatic Fallback**: Agent automatically switched to backup service
3. ✅ **Graceful Degradation**: User experience remained smooth despite failure
4. ✅ **Transparent Communication**: Agent informed user about using backup data

**How It Works:**
- Primary tool returns error message instead of data
- Agent's instructions contain explicit error handling logic
- LLM reasons about error and selects backup tool
- Backup tool provides alternate data source

**Real-World Applications:**
- 🌐 **API Redundancy**: Multiple API providers for critical services
- 💾 **Cache Fallback**: Use cached data when live APIs fail
- 🔄 **Retry Logic**: Automatic retries with exponential backoff
- 📊 **Service Monitoring**: Track which services are failing

**Production Best Practices:**
- Implement circuit breakers for failing services
- Log all errors for monitoring and alerting
- Provide clear user feedback about degraded service
- Test failure scenarios regularly
- Monitor backup service usage patterns

**Next Steps:**
- Add logging and monitoring (Azure Application Insights)
- Implement rate limiting and throttling
- Add health checks for all services
- Build dashboards for error tracking

---

**🎉 Congratulations!** You've built a production-ready AI agent with robust error handling using Microsoft Agent Framework!